<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-dqn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning para _cartpole_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Basado en:
> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## Configuración

In [ ]:
%pip install "gymnasium[classic-control]>=1.1"

In [ ]:
# modo de renderizado 'rgb_array' (cambiar a 'human' para ejecución local con pantalla)
render_mode = "rgb_array"

In [ ]:
# comprobar si existen los ficheros de apoyo
import os.path
if not os.path.isfile('dqn_agent.py'):
  !git clone -b es https://github.com/jgromero/drl-csic
  %cd drl-csic/code/from-scratch

## Entorno

Usaremos [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), un problema de control sencillo.

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cartpole.gif?raw=true"/>


Los espacios de acciones y de observaciones se pueden mostrar como sigue.

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

Podemos interactuar con el entorno con un agente sin entrenar usando la clase `Agent` de [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py).

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=4, action_size=2, seed=0)

state, _ = env.reset()

for j in range(500):
    action = agent_untrained.act(state)
    env.render()
    state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

env.close()

## Algoritmo DQN
A continuación proporcionamos una implementación genérica de Deep Q-Learning (DQN). Nos apoyamos en ficheros externos:

*   [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py): Implementación de las capacidades del agente (`act`, `step`, etc.)
*   [model.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/model.py): Implementación de la red neuronal del agente (la red Q)

El entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) se considera resuelto cuando el agente obtiene una puntuación media de al menos $475$ puntos en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la puntuación máxima.

In [ ]:
import random
import torch
import numpy as np
from collections import deque
from dqn_agent import device
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
SOLVED_SCORE = 475.0   # puntuación media (últimos 100 episodios) para considerar resuelto CartPole-v1

def dqn(agent, n_episodes=2000, max_t=500, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        max_t (int): número máximo de pasos por episodio
        eps_start (float): valor inicial de epsilon
        eps_end (float): valor final de epsilon
        eps_decay (float): factor de multiplicación de epsilon (por episodio)
        print_every (int): mostrar la puntuación cada print_every episodios
    """
    scores = []                        # puntuaciones de todos los episodios
    scores_window = deque(maxlen=100)  # puntuaciones de los últimos 100 episodios
    eps = eps_start                    # inicializar epsilon

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # elegir la acción At con la política ε-greedy
            action = agent.act(state, eps)

            # aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # almacenar <St, At, Rt+1, St+1>, entrenar y actualizar
            # (solo 'terminated' marca St+1 como terminal; tras un truncamiento hay que seguir haciendo bootstrapping)
            agent.step(state, action, reward, next_state, terminated)

            # avanzar al siguiente estado
            state = next_state
            score += reward

            if terminated or truncated:
                break

        scores_window.append(score)       # guardar la última puntuación (ventana de 100)
        scores.append(score)              # guardar la última puntuación (todos los episodios)
        eps = max(eps_end, eps_decay*eps) # reducir epsilon

        # mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print('Episodio {}\tPuntuación media (últimos 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\n¡Problema resuelto en {:d} episodios!\tPuntuación media (últimos 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblema no resuelto en {:d} episodios.\tPuntuación media (últimos 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.qnetwork_local.state_dict(), 'checkpoint-cartpole-dqn.pth') # guardar los pesos del agente entrenado

    return scores

agent = Agent(state_size=4, action_size=2, seed=0)
scores = dqn(agent)

# representar las puntuaciones
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Puntuación')
plt.xlabel('Episodio nº')
plt.show()

## Visualizar el agente entrenado

Posteriormente podemos cargar los pesos en el agente y visualizar cómo actúa en el entorno. (Se requiere ejecución local para el renderizado 2D, no Google Colaboratory.)

In [ ]:
# cargar los pesos desde `checkpoint-cartpole-dqn.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load('checkpoint-cartpole-dqn.pth', map_location=device, weights_only=True))

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action = agent_load.act(state)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()